In [26]:
import joblib
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [27]:
df = pd.read_csv("Auto.csv").dropna().copy()

FEATURE_COLUMNS = [
    "mpg",
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "year",
    "origin",
]
CATEGORICAL_COLS = ["origin"]
NUMERICAL_COLS = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_COLS]

X_raw = df[FEATURE_COLUMNS]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), NUMERICAL_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ]
)

In [31]:
help(GaussianHMM.fit)

Help on function fit in module hmmlearn.base:

fit(self, X, lengths=None)
    Estimate model parameters.
    
    An initialization step is performed before entering the
    EM algorithm. If you want to avoid this step for a subset of
    the parameters, pass proper ``init_params`` keyword argument
    to estimator's constructor.
    
    Parameters
    ----------
    X : array-like, shape (n_samples, n_features)
        Feature matrix of individual samples.
    lengths : array-like of integers, shape (n_sequences, )
        Lengths of the individual sequences in ``X``. The sum of
        these should be ``n_samples``.
    
    Returns
    -------
    self : object
        Returns self.



In [28]:

X = preprocessor.fit_transform(X_raw)

N_STATES = 4
hmm = GaussianHMM(
    n_components=N_STATES,
    covariance_type="diag",
    n_iter=300,
    random_state=42,
    verbose=True,
    init_params="stmc",
    params="stmc",
)

hmm.fit(X)

hidden_states = hmm.predict(X)
log_likelihood = hmm.score(X)

df["hidden_state"] = hidden_states
print("Learned transition matrix:\n", hmm.transmat_)


Learned transition matrix:
 [[0.49180718 0.10119861 0.20100748 0.20598673]
 [0.01459876 0.63605179 0.07043065 0.2789188 ]
 [0.04810795 0.09910192 0.77206903 0.0807211 ]
 [0.06926807 0.34153698 0.05079563 0.53839932]]


         1   -4166.06660312             +nan
         2   -1647.15800705   +2518.90859606
         3     -23.66070156   +1623.49730549
         4     502.59160655    +526.25230812
         5    1031.96347665    +529.37187009
         6    1127.78580851     +95.82233186
         7    1350.91570701    +223.12989850
         8    1378.05029939     +27.13459238
         9    1394.50734858     +16.45704920
        10    1416.75325704     +22.24590846
        11    1434.50350739     +17.75025035
        12    1442.51377878      +8.01027139
        13    1444.24907788      +1.73529909
        14    1444.86293965      +0.61386177
        15    1445.13059262      +0.26765297
        16    1445.26082775      +0.13023514
        17    1445.33044072      +0.06961297
        18    1445.37007261      +0.03963189
        19    1445.39354262      +0.02347000
        20    1445.40778596      +0.01424334
        21    1445.41656294      +0.00877698


In [33]:
print("Learned transition matrix:\n", hmm.covars_prior)

Learned transition matrix:
 0.01


In [32]:

joblib.dump(
    {
        "model": hmm,
        "preprocessor": preprocessor,
        "feature_columns": FEATURE_COLUMNS,
        "categorical_cols": CATEGORICAL_COLS,
        "numerical_cols": NUMERICAL_COLS,
    },
    "baum_welch_hmm.joblib",
)

df[["mpg", "horsepower", "weight", "origin", "hidden_state"]].head(10)

,mpg,horsepower,weight,origin,hidden_state
0,18.0,130,3504,1,2
1,15.0,165,3693,1,2
2,18.0,150,3436,1,2
3,16.0,150,3433,1,2
4,17.0,140,3449,1,2
5,15.0,198,4341,1,2
6,14.0,220,4354,1,2
7,14.0,215,4312,1,2
8,14.0,225,4425,1,2
9,15.0,190,3850,1,2
